In [1]:
import pandas as pd

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

train_traffic = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/train/traffic.csv")
train_schedule = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/train/schedule.csv")
train_labels = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/labels/labels_train.csv")

test_traffic = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/test/traffic.csv")
test_schedule = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/test/schedule.csv")
test_labels = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/labels/labels_test.csv")

Mounted at /content/drive


/tmp/ipykernel_533/269612624.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  train_traffic = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MT_Hack_2026/train/traffic.csv")


In [2]:
# зависимости (в Colab обычно уже есть; ставим тихо при необходимости)
import importlib, subprocess, sys
for _mod, _pkg in [('lightgbm', 'lightgbm'), ('catboost', 'catboost')]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])


In [3]:
# ---- подготовка: datetime/числовые колонки к единому виду (как prepare_* в fe_lib) ----
import numpy as np
import pandas as pd

def _norm_traffic(df):
    df = df.copy()
    for c in ('event_time', 'gps_time', 'receive_time'):
        df[c] = pd.to_datetime(df[c], errors='coerce')
    for c in ('lon', 'lat', 'alt', 'speed', 'heading'):
        df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in ('location_valid', 'is_hist_data'):
        df[c] = df[c].astype(str).str.strip().isin(['True', 'true', '1'])
    return df

def _norm_schedule(df):
    df = df.copy()
    df['time_begin'] = pd.to_datetime(df['time_begin'], errors='coerce')
    return df

def _norm_points(df):
    df = df.copy()
    df['T'] = pd.to_datetime(df['T'], errors='coerce')
    df['target_time_begin'] = pd.to_datetime(df['target_time_begin'], errors='coerce')
    return df

train_traffic  = _norm_traffic(train_traffic)
test_traffic   = _norm_traffic(test_traffic)
train_schedule = _norm_schedule(train_schedule)
test_schedule  = _norm_schedule(test_schedule)
train_labels   = _norm_points(train_labels)
test_labels    = _norm_points(test_labels)
print('traffic:', train_traffic.shape, test_traffic.shape,
      '| schedule:', train_schedule.shape, test_schedule.shape,
      '| labels:', train_labels.shape, test_labels.shape)


traffic: (287849, 14) (105945, 14) | schedule: (16674, 8) (5558, 8) | labels: (4434, 8) (353, 8)


In [4]:
# ===================== FEATURE ENGINEERING (fe_lib) =====================
"""Feature engineering library for the MosTrans delay-prediction task.

All features are computed strictly from information available at the forecast
moment T (telemetry with ``available_time = max(event_time, receive_time) <= T``,
plus schedule *plan* rows and the ``cur_dev_s`` hint).  ``time_fact_begin`` is
never read -- it only exists inside labels to build the target.

Public entry point: :func:`build_features`.
"""


import numpy as np
import pandas as pd

import argparse
import time

SEED = 143
WINDOWS = (1, 3, 5, 10, 15, 30)          # telemetry aggregation windows, minutes
RADII = (300, 500, 1000, 2000)           # neighbourhood radii, meters
SPEED_CAP = 120.0                        # sensor anomaly cap, km/h
ALT_MIN, ALT_MAX = 0.0, 1000.0
EARTH_R = 6_371_000.0

META_COLS = ["sample_id", "tr_id", "T", "target_stop_id", "target_time_begin"]


# --------------------------------------------------------------------------- #
# geometry helpers
# --------------------------------------------------------------------------- #
def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))


def bearing_deg(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def angle_diff_deg(a, b):
    return ((a - b + 180) % 360) - 180


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


def linear_slope_per_min(seconds, values):
    mask = np.isfinite(values)
    if mask.sum() < 2:
        return np.nan
    t = seconds[mask] / 60.0
    y = values[mask]
    t = t - t.mean()
    denom = float(np.dot(t, t))
    return float(np.dot(t, y - y.mean()) / denom) if denom > 0 else 0.0


# --------------------------------------------------------------------------- #
# preparation
# --------------------------------------------------------------------------- #
SCHEDULE_USECOLS = [
    "tt_action_item_id", "time_begin", "manual_fill", "tr_id", "geom", "building_address",
]


def prepare_traffic(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    for col in ("event_time", "gps_time", "receive_time"):
        x[col] = pd.to_datetime(x[col], errors="coerce")
    for col in ("lon", "lat", "alt", "speed", "heading"):
        x[col] = pd.to_numeric(x[col], errors="coerce")
    x["location_valid"] = x["location_valid"].astype(bool)
    x["is_hist_data"] = x["is_hist_data"].astype(bool)

    x["available_time"] = x[["event_time", "receive_time"]].max(axis=1)
    x["receive_lag_s"] = (x.receive_time - x.event_time).dt.total_seconds()
    x["gps_event_lag_s"] = (x.event_time - x.gps_time).dt.total_seconds()
    x["speed_outlier"] = x.speed > SPEED_CAP
    x["alt_outlier"] = (x.alt < ALT_MIN) | (x.alt > ALT_MAX)
    x["speed_clean"] = x.speed.clip(ALT_MIN, SPEED_CAP)
    x["alt_clean"] = x.alt.clip(ALT_MIN, ALT_MAX)
    x["valid_coords"] = x.location_valid & x.lon.notna() & x.lat.notna()
    return x.sort_values(["tr_id", "available_time"]).reset_index(drop=True)


def prepare_schedule(df: pd.DataFrame) -> pd.DataFrame:
    """Schedule *plan* rows only (no fact columns are ever touched)."""
    x = df.copy()
    x["time_begin"] = pd.to_datetime(x["time_begin"], errors="coerce")
    x["tt_action_item_id"] = pd.to_numeric(x["tt_action_item_id"], errors="coerce").astype("Int64")
    x["tr_id"] = pd.to_numeric(x["tr_id"], errors="coerce").astype("int64")
    x["manual_fill"] = x["manual_fill"].astype(bool)

    coords = x.geom.astype("string").str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
    x["stop_lon"], x["stop_lat"] = coords[0], coords[1]
    x["stop_key"] = x.stop_lon.round(5).astype("string") + "_" + x.stop_lat.round(5).astype("string")
    x["address_missing"] = x.building_address.isna().astype("int8")

    x = x.sort_values(["tr_id", "time_begin"]).reset_index(drop=True)
    by_vehicle = x.groupby("tr_id", sort=False)
    x["schedule_pos"] = by_vehicle.cumcount()
    x["schedule_rows"] = by_vehicle.tr_id.transform("size")
    x["schedule_progress"] = x.schedule_pos / (x.schedule_rows - 1).replace(0, np.nan)
    x["planned_gap_prev_s"] = by_vehicle.time_begin.diff().dt.total_seconds()
    x["planned_gap_next_s"] = (by_vehicle.time_begin.shift(-1) - x.time_begin).dt.total_seconds()

    x["prev_stop_lon"] = by_vehicle.stop_lon.shift(1)
    x["prev_stop_lat"] = by_vehicle.stop_lat.shift(1)
    x["next_stop_lon"] = by_vehicle.stop_lon.shift(-1)
    x["next_stop_lat"] = by_vehicle.stop_lat.shift(-1)
    x["prev_segment_m"] = haversine_m(x.prev_stop_lon, x.prev_stop_lat, x.stop_lon, x.stop_lat)
    x["next_segment_m"] = haversine_m(x.stop_lon, x.stop_lat, x.next_stop_lon, x.next_stop_lat)
    x["planned_speed_prev_kmh"] = 3.6 * x.prev_segment_m / x.planned_gap_prev_s.replace(0, np.nan)
    x["planned_speed_next_kmh"] = 3.6 * x.next_segment_m / x.planned_gap_next_s.replace(0, np.nan)

    x["stop_visit_count"] = x.groupby("stop_key").stop_key.transform("size")

    stop_time = x[["tt_action_item_id", "stop_key", "time_begin"]].sort_values(
        ["stop_key", "time_begin"]
    ).copy()
    by_stop = stop_time.groupby("stop_key", sort=False)
    stop_time["stop_headway_prev_s"] = by_stop.time_begin.diff().dt.total_seconds()
    stop_time["stop_headway_next_s"] = (
        by_stop.time_begin.shift(-1) - stop_time.time_begin
    ).dt.total_seconds()
    return x.merge(
        stop_time[["tt_action_item_id", "stop_headway_prev_s", "stop_headway_next_s"]],
        on="tt_action_item_id", how="left",
    )


def prepare_points(df: pd.DataFrame) -> pd.DataFrame:
    p = df.copy()
    p["T"] = pd.to_datetime(p["T"], errors="coerce")
    p["target_time_begin"] = pd.to_datetime(p["target_time_begin"], errors="coerce")
    p["tr_id"] = pd.to_numeric(p["tr_id"], errors="coerce").astype("int64")
    p["target_stop_id"] = pd.to_numeric(p["target_stop_id"], errors="coerce").astype("Int64")
    p["cur_dev_s"] = pd.to_numeric(p["cur_dev_s"], errors="coerce")
    return p


# --------------------------------------------------------------------------- #
# base / point-level features
# --------------------------------------------------------------------------- #
SCHED_TARGET_COLS = [
    "tt_action_item_id", "manual_fill", "stop_lon", "stop_lat", "stop_key",
    "building_address", "address_missing", "schedule_pos", "schedule_rows",
    "schedule_progress", "planned_gap_prev_s", "planned_gap_next_s",
    "prev_segment_m", "next_segment_m", "planned_speed_prev_kmh",
    "planned_speed_next_kmh", "stop_visit_count", "stop_headway_prev_s",
    "stop_headway_next_s", "prev_stop_lon", "prev_stop_lat",
    "next_stop_lon", "next_stop_lat",
]


def add_base_features(points: pd.DataFrame, schedule: pd.DataFrame) -> pd.DataFrame:
    p = points.copy()
    p = p.merge(schedule[SCHED_TARGET_COLS], left_on="target_stop_id",
                right_on="tt_action_item_id", how="left")
    p = p.rename(columns={"building_address": "target_address"})

    p["horizon_s"] = (p.target_time_begin - p["T"]).dt.total_seconds()
    p["horizon_min"] = p.horizon_s / 60.0
    p["minute_of_day"] = p["T"].dt.hour * 60 + p["T"].dt.minute + p["T"].dt.second / 60.0
    p["target_minute_of_day"] = (
        p.target_time_begin.dt.hour * 60 + p.target_time_begin.dt.minute
        + p.target_time_begin.dt.second / 60.0
    )
    two_pi = 2 * np.pi
    p["time_sin"] = np.sin(two_pi * p.minute_of_day / 1440)
    p["time_cos"] = np.cos(two_pi * p.minute_of_day / 1440)
    p["target_time_sin"] = np.sin(two_pi * p.target_minute_of_day / 1440)
    p["target_time_cos"] = np.cos(two_pi * p.target_minute_of_day / 1440)
    p["hour"] = p["T"].dt.hour
    p["target_hour"] = p.target_time_begin.dt.hour
    p["dow"] = p["T"].dt.dayofweek

    cd = p.cur_dev_s
    p["cur_dev_abs"] = cd.abs()
    p["cur_dev_sign"] = np.sign(cd)
    p["cur_dev_log"] = signed_log1p(cd)
    p["cur_dev_sqrt"] = np.sign(cd) * np.sqrt(cd.abs())
    p["cur_dev_sq"] = cd**2
    p["cur_dev_per_min"] = cd / p.horizon_min
    p["manual_fill"] = p.manual_fill.astype("int8")
    for dec in (2, 3, 4):
        p[f"target_grid_{dec}"] = (
            p.stop_lon.round(dec).astype("string") + "_" + p.stop_lat.round(dec).astype("string")
        )
    return p


def add_latest_state(features: pd.DataFrame, traffic: pd.DataFrame) -> pd.DataFrame:
    p = features.copy()
    left = p[["sample_id", "tr_id", "T"]].sort_values("T")

    any_cols = [
        "tr_id", "available_time", "event_time", "receive_time", "gps_time",
        "lon", "lat", "speed_clean", "alt_clean", "heading", "location_valid",
        "is_hist_data", "receive_lag_s", "gps_event_lag_s", "speed_outlier", "alt_outlier",
    ]
    any_state = pd.merge_asof(
        left, traffic[any_cols].sort_values("available_time"),
        left_on="T", right_on="available_time", by="tr_id", direction="backward",
    ).set_index("sample_id")
    any_state = any_state.rename(columns={
        "available_time": "last_packet_time", "event_time": "last_event_time",
        "receive_time": "last_receive_time", "gps_time": "last_gps_time",
        "lon": "last_lon_any", "lat": "last_lat_any",
        "speed_clean": "last_speed_kmh", "alt_clean": "last_alt_m",
        "heading": "last_heading_deg", "location_valid": "last_location_valid",
        "is_hist_data": "last_is_hist", "receive_lag_s": "last_receive_lag_s",
        "gps_event_lag_s": "last_gps_event_lag_s", "speed_outlier": "last_speed_outlier",
        "alt_outlier": "last_alt_outlier",
    })
    for name, col in [("packet_age_s", "last_packet_time"), ("event_age_s", "last_event_time"),
                      ("receive_age_s", "last_receive_time")]:
        any_state[name] = (any_state["T"] - any_state[col]).dt.total_seconds()

    valid = traffic.loc[
        traffic.valid_coords,
        ["tr_id", "available_time", "lon", "lat", "speed_clean", "alt_clean", "heading"],
    ]
    valid_state = pd.merge_asof(
        left, valid.sort_values("available_time"),
        left_on="T", right_on="available_time", by="tr_id", direction="backward",
    ).set_index("sample_id").rename(columns={
        "available_time": "last_valid_time", "lon": "last_valid_lon", "lat": "last_valid_lat",
        "speed_clean": "last_valid_speed_kmh", "alt_clean": "last_valid_alt_m",
        "heading": "last_valid_heading_deg",
    })
    valid_state["valid_age_s"] = (valid_state["T"] - valid_state.last_valid_time).dt.total_seconds()

    p = p.set_index("sample_id")
    p = p.join(any_state.drop(columns=["tr_id", "T"]))
    p = p.join(valid_state.drop(columns=["tr_id", "T"]))
    return p.reset_index()


def add_physical_features(features: pd.DataFrame) -> pd.DataFrame:
    x = features.copy()
    x["distance_to_target_m"] = haversine_m(
        x.last_valid_lon, x.last_valid_lat, x.stop_lon, x.stop_lat)
    x["bearing_to_target_deg"] = bearing_deg(
        x.last_valid_lon, x.last_valid_lat, x.stop_lon, x.stop_lat)
    x["heading_error_deg"] = angle_diff_deg(x.last_valid_heading_deg, x.bearing_to_target_deg)
    x["heading_error_abs_deg"] = x.heading_error_deg.abs()
    x["heading_alignment"] = np.cos(np.radians(x.heading_error_deg))
    x["last_heading_sin"] = np.sin(np.radians(x.last_valid_heading_deg))
    x["last_heading_cos"] = np.cos(np.radians(x.last_valid_heading_deg))
    x["bearing_sin"] = np.sin(np.radians(x.bearing_to_target_deg))
    x["bearing_cos"] = np.cos(np.radians(x.bearing_to_target_deg))

    x["projected_speed_kmh"] = x.last_valid_speed_kmh * x.heading_alignment
    x["required_speed_kmh"] = 3.6 * x.distance_to_target_m / x.horizon_s
    x["speed_margin_kmh"] = x.last_valid_speed_kmh - x.required_speed_kmh
    x["projected_speed_margin_kmh"] = x.projected_speed_kmh - x.required_speed_kmh
    x["speed_to_required_ratio"] = x.last_valid_speed_kmh / x.required_speed_kmh.replace(0, np.nan)
    x["eta_last_speed_s"] = 3.6 * x.distance_to_target_m / x.last_valid_speed_kmh.replace(0, np.nan)
    x["eta_projected_speed_s"] = (
        3.6 * x.distance_to_target_m / x.projected_speed_kmh.where(x.projected_speed_kmh > 0))
    x["eta_margin_last_speed_s"] = x.horizon_s - x.eta_last_speed_s
    x["eta_margin_projected_s"] = x.horizon_s - x.eta_projected_speed_s
    x["distance_prev_stop_m"] = haversine_m(
        x.last_valid_lon, x.last_valid_lat, x.prev_stop_lon, x.prev_stop_lat)
    x["distance_next_stop_m"] = haversine_m(
        x.last_valid_lon, x.last_valid_lat, x.next_stop_lon, x.next_stop_lat)
    for dec in (2, 3, 4):
        x[f"current_grid_{dec}"] = (
            x.last_valid_lon.round(dec).astype("string") + "_"
            + x.last_valid_lat.round(dec).astype("string"))
    return x


# --------------------------------------------------------------------------- #
# schedule path features
# --------------------------------------------------------------------------- #
def make_schedule_groups(schedule):
    return {tid: g.sort_values("time_begin").reset_index(drop=True)
            for tid, g in schedule.groupby("tr_id")}


def add_schedule_path_features(features, schedule_groups):
    rows = []
    cols = ["sample_id", "tr_id", "T", "target_time_begin"]
    for sample_id, tr_id, T, target_time in features[cols].itertuples(index=False, name=None):
        sched = schedule_groups.get(tr_id)
        out = {"sample_id": sample_id, "scheduled_stops_to_target": 0}
        if sched is None or sched.empty:
            rows.append(out)
            continue
        times = sched.time_begin.to_numpy(dtype="datetime64[ns]")
        left = np.searchsorted(times, pd.Timestamp(T).to_datetime64(), side="right")
        right = np.searchsorted(times, pd.Timestamp(target_time).to_datetime64(), side="right")
        future = sched.iloc[left:right]
        out["scheduled_stops_to_target"] = len(future)
        if len(future):
            lon = future.stop_lon.to_numpy(float)
            lat = future.stop_lat.to_numpy(float)
            gaps = future.time_begin.diff().dt.total_seconds().dropna().to_numpy()
            out["first_future_stop_lon"] = lon[0]
            out["first_future_stop_lat"] = lat[0]
            out["scheduled_path_m"] = (
                haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:]).sum() if len(future) > 1 else 0.0)
            out["future_gap_mean_s"] = gaps.mean() if len(gaps) else np.nan
            out["future_gap_max_s"] = gaps.max() if len(gaps) else np.nan
            out["future_manual_fill_share"] = future.manual_fill.mean()
        rows.append(out)

    extra = pd.DataFrame(rows).set_index("sample_id")
    x = features.set_index("sample_id").join(extra).reset_index()
    x["distance_to_first_future_stop_m"] = haversine_m(
        x.last_valid_lon, x.last_valid_lat, x.first_future_stop_lon, x.first_future_stop_lat)
    x["route_proxy_m"] = x.distance_to_first_future_stop_m + x.scheduled_path_m
    x["route_proxy_required_speed_kmh"] = 3.6 * x.route_proxy_m / x.horizon_s
    x["straight_to_route_ratio"] = x.distance_to_target_m / x.route_proxy_m.replace(0, np.nan)
    return x


# --------------------------------------------------------------------------- #
# windowed telemetry features
# --------------------------------------------------------------------------- #
def make_traffic_groups(traffic):
    groups = {}
    for tr_id, g in traffic.groupby("tr_id", sort=False):
        groups[tr_id] = {
            "available": g.available_time.to_numpy(dtype="datetime64[ns]").astype("int64"),
            "event": g.event_time.to_numpy(dtype="datetime64[ns]").astype("int64"),
            "speed": g.speed_clean.to_numpy(float),
            "alt": g.alt_clean.to_numpy(float),
            "receive_lag": g.receive_lag_s.to_numpy(float),
            "gps_lag": g.gps_event_lag_s.to_numpy(float),
            "valid": g.valid_coords.to_numpy(bool),
            "hist": g.is_hist_data.to_numpy(bool),
            "speed_outlier": g.speed_outlier.to_numpy(bool),
            "alt_outlier": g.alt_outlier.to_numpy(bool),
            "lon": g.lon.to_numpy(float),
            "lat": g.lat.to_numpy(float),
            "heading": g.heading.to_numpy(float),
        }
    return groups


def window_features(points, traffic_groups, minutes):
    rows = []
    delta_ns = int(minutes * 60 * 1e9)
    cols = ["sample_id", "tr_id", "T", "stop_lon", "stop_lat"]
    prefix = f"w{minutes}m_"

    for sample_id, tr_id, T, target_lon, target_lat in points[cols].itertuples(index=False, name=None):
        group = traffic_groups.get(tr_id)
        out = {"sample_id": sample_id, prefix + "n": 0}
        if group is None:
            rows.append(out)
            continue
        end = pd.Timestamp(T).value
        start = end - delta_ns
        left = np.searchsorted(group["available"], start, side="left")
        right = np.searchsorted(group["available"], end, side="right")
        sl = slice(left, right)
        n = right - left
        out[prefix + "n"] = n
        if n == 0:
            rows.append(out)
            continue

        available = group["available"][sl]
        speed = group["speed"][sl]
        valid = group["valid"][sl]

        span = max((available[-1] - available[0]) / 1e9, 0)
        elapsed = (available - available[0]) / 1e9
        out[prefix + "span_s"] = span
        out[prefix + "packets_per_min"] = n / minutes
        out[prefix + "valid_share"] = valid.mean()
        out[prefix + "hist_share"] = group["hist"][sl].mean()
        out[prefix + "speed_outlier_share"] = group["speed_outlier"][sl].mean()
        out[prefix + "alt_outlier_share"] = group["alt_outlier"][sl].mean()

        speed_ok = speed[np.isfinite(speed)]
        if len(speed_ok):
            q10, q25, q50, q75, q90 = np.quantile(speed_ok, [.10, .25, .50, .75, .90])
            out.update({
                prefix + "speed_mean": speed_ok.mean(), prefix + "speed_median": q50,
                prefix + "speed_std": speed_ok.std(), prefix + "speed_min": speed_ok.min(),
                prefix + "speed_max": speed_ok.max(), prefix + "speed_q10": q10,
                prefix + "speed_q90": q90, prefix + "speed_iqr": q75 - q25,
                prefix + "stopped_share": (speed_ok <= 1).mean(),
                prefix + "slow_share": (speed_ok <= 5).mean(),
                prefix + "fast_share": (speed_ok >= 40).mean(),
                prefix + "speed_first": speed_ok[0], prefix + "speed_last": speed_ok[-1],
                prefix + "speed_delta": speed_ok[-1] - speed_ok[0],
                prefix + "speed_slope": linear_slope_per_min(elapsed, speed),
            })
            if len(speed_ok) > 1:
                out[prefix + "speed_change_abs_mean"] = np.abs(np.diff(speed_ok)).mean()
                out[prefix + "stop_transitions"] = int(((speed_ok[:-1] > 1) & (speed_ok[1:] <= 1)).sum())

        valid_speed = speed[valid & np.isfinite(speed)]
        if len(valid_speed):
            out[prefix + "valid_speed_mean"] = valid_speed.mean()
            out[prefix + "valid_speed_median"] = np.median(valid_speed)

        alt = group["alt"][sl]
        alt = alt[np.isfinite(alt)]
        if len(alt):
            q25, q50, q75 = np.quantile(alt, [.25, .50, .75])
            out[prefix + "alt_median"] = q50
            out[prefix + "alt_iqr"] = q75 - q25

        rl = group["receive_lag"][sl]
        rl = rl[np.isfinite(rl)]
        if len(rl):
            out[prefix + "receive_lag_median"] = np.median(rl)
            out[prefix + "receive_lag_p95"] = np.quantile(rl, .95)
            out[prefix + "receive_lag_max"] = rl.max()
            out[prefix + "receive_lag_negative_share"] = (rl < 0).mean()

        gl = group["gps_lag"][sl]
        gl = gl[np.isfinite(gl)]
        if len(gl):
            out[prefix + "gps_event_lag_median"] = np.median(gl)
            out[prefix + "gps_event_lag_abs_median"] = np.median(np.abs(gl))

        event = group["event"][sl]
        if len(event) > 1:
            gaps = np.diff(event) / 1e9
            out[prefix + "event_gap_median"] = np.median(gaps)
            out[prefix + "event_gap_p95"] = np.quantile(gaps, .95)
            out[prefix + "event_gap_max"] = gaps.max()
            out[prefix + "event_time_backwards_share"] = (gaps < 0).mean()

        heading = group["heading"][sl]
        heading = heading[np.isfinite(heading)]
        if len(heading):
            rad = np.radians(heading)
            out[prefix + "heading_resultant"] = np.hypot(np.sin(rad).mean(), np.cos(rad).mean())

        valid_idx = np.flatnonzero(valid & np.isfinite(group["lon"][sl]) & np.isfinite(group["lat"][sl]))
        if len(valid_idx):
            lon = group["lon"][sl][valid_idx]
            lat = group["lat"][sl][valid_idx]
            vtime = available[valid_idx]
            out[prefix + "valid_n"] = len(valid_idx)
            distance = haversine_m(lon, lat, target_lon, target_lat)
            out[prefix + "target_distance_first_m"] = distance[0]
            out[prefix + "target_distance_last_m"] = distance[-1]
            out[prefix + "target_progress_m"] = distance[0] - distance[-1]
            if len(valid_idx) > 1:
                steps = haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:])
                path = steps.sum()
                displacement = haversine_m(lon[0], lat[0], lon[-1], lat[-1])
                valid_span = (vtime[-1] - vtime[0]) / 1e9
                out[prefix + "path_m"] = path
                out[prefix + "displacement_m"] = displacement
                out[prefix + "tortuosity"] = path / max(displacement, 1)
                out[prefix + "step_median"] = np.median(steps)
                out[prefix + "step_p95"] = np.quantile(steps, .95)
                if valid_span > 0:
                    out[prefix + "trajectory_speed_kmh"] = 3.6 * path / valid_span
                    out[prefix + "target_progress_kmh"] = 3.6 * out[prefix + "target_progress_m"] / valid_span
        rows.append(out)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# neighbour (other vehicles) context
# --------------------------------------------------------------------------- #
def make_valid_groups(traffic):
    return {
        tr_id: g.loc[g.valid_coords, ["available_time", "lon", "lat", "speed_clean"]]
        .sort_values("available_time").reset_index(drop=True)
        for tr_id, g in traffic.groupby("tr_id", sort=False)
    }


def neighbour_features(features, valid_groups, radii=RADII):
    rows = []
    for T, points_t in features.groupby("T", sort=False):
        t = pd.Timestamp(T).to_datetime64()
        snap_lon, snap_lat, snap_speed, snap_tid = [], [], [], []
        for tr_id, group in valid_groups.items():
            times = group.available_time.to_numpy(dtype="datetime64[ns]")
            idx = np.searchsorted(times, t, side="right") - 1
            if idx >= 0:
                row = group.iloc[idx]
                snap_tid.append(tr_id)
                snap_lon.append(row.lon)
                snap_lat.append(row.lat)
                snap_speed.append(row.speed_clean)
        snapshot = pd.DataFrame({"tr_id": snap_tid, "lon": snap_lon, "lat": snap_lat,
                                 "speed": snap_speed})
        city_speed = snapshot.speed.dropna()
        snap_lon_np = snapshot.lon.to_numpy(float)
        snap_lat_np = snapshot.lat.to_numpy(float)
        speed_np = snapshot.speed.to_numpy(float)

        for row in points_t.itertuples(index=False):
            out = {
                "sample_id": row.sample_id,
                "snapshot_vehicle_n": len(snapshot),
                "city_speed_mean": city_speed.mean(),
                "city_speed_median": city_speed.median(),
                "city_slow_share": (city_speed <= 5).mean() if len(city_speed) else np.nan,
            }
            other = snapshot.tr_id.ne(row.tr_id).to_numpy()
            d_cur = haversine_m(snap_lon_np, snap_lat_np, row.last_valid_lon, row.last_valid_lat)
            d_tgt = haversine_m(snap_lon_np, snap_lat_np, row.stop_lon, row.stop_lat)
            for radius in radii:
                for name, dist in (("local", d_cur), ("target_area", d_tgt)):
                    mask = other & (dist <= radius)
                    out[f"{name}_vehicles_{radius}m"] = int(mask.sum())
                    nearby = speed_np[mask]
                    nearby = nearby[np.isfinite(nearby)]
                    if len(nearby):
                        out[f"{name}_speed_mean_{radius}m"] = nearby.mean()
                        out[f"{name}_speed_median_{radius}m"] = np.median(nearby)
                        out[f"{name}_slow_share_{radius}m"] = (nearby <= 5).mean()
            rows.append(out)
    extra = pd.DataFrame(rows).set_index("sample_id")
    return features.set_index("sample_id").join(extra).reset_index()


# --------------------------------------------------------------------------- #
# interactions / crosses
# --------------------------------------------------------------------------- #
def add_cross_features(features):
    x = features.copy()
    x["valid_age_gt_30s"] = (x.valid_age_s > 30).astype("int8")
    x["valid_age_gt_90s"] = (x.valid_age_s > 90).astype("int8")
    x["valid_age_gt_5m"] = (x.valid_age_s > 300).astype("int8")
    x["packet_age_gt_30s"] = (x.packet_age_s > 30).astype("int8")

    for short, long in [(1, 5), (3, 15), (5, 15), (5, 30), (10, 30)]:
        x[f"speed_mean_{short}m_minus_{long}m"] = x[f"w{short}m_speed_mean"] - x[f"w{long}m_speed_mean"]
        x[f"slow_share_{short}m_minus_{long}m"] = x[f"w{short}m_slow_share"] - x[f"w{long}m_slow_share"]
        x[f"valid_share_{short}m_minus_{long}m"] = x[f"w{short}m_valid_share"] - x[f"w{long}m_valid_share"]
        x[f"progress_{short}m_minus_{long}m"] = (
            x[f"w{short}m_target_progress_kmh"] - x[f"w{long}m_target_progress_kmh"])

    x["speed_recent_vs_required"] = x.w5m_speed_mean - x.required_speed_kmh
    x["progress_recent_vs_required"] = x.w5m_target_progress_kmh - x.required_speed_kmh
    x["speed_15m_vs_required"] = x.w15m_speed_mean - x.required_speed_kmh

    x["cur_dev_x_horizon"] = x.cur_dev_s * x.horizon_min
    x["cur_dev_x_required_speed"] = x.cur_dev_s * x.required_speed_kmh
    x["cur_dev_x_distance_km"] = x.cur_dev_s * x.distance_to_target_m / 1000
    x["cur_dev_per_stop"] = x.cur_dev_s / x.scheduled_stops_to_target.replace(0, np.nan)
    x["cur_dev_per_km"] = x.cur_dev_s / (x.route_proxy_m / 1000).replace(0, np.nan)
    x["planned_seconds_per_stop"] = x.horizon_s / x.scheduled_stops_to_target.replace(0, np.nan)
    x["route_m_per_stop"] = x.route_proxy_m / x.scheduled_stops_to_target.replace(0, np.nan)
    x["scheduled_stops_per_min"] = x.scheduled_stops_to_target / x.horizon_min

    x["cur_dev_plus_eta_margin"] = x.cur_dev_s + x.eta_margin_last_speed_s
    x["route_speed_margin_kmh"] = x.w5m_speed_mean - x.route_proxy_required_speed_kmh
    x["planned_vs_straight_distance_m"] = x.route_proxy_m - x.distance_to_target_m
    x["route_straightness"] = x.distance_to_target_m / x.route_proxy_m.replace(0, np.nan)
    x["required_vs_w5_speed_ratio"] = x.required_speed_kmh / x.w5m_speed_mean.replace(0, np.nan)
    x["route_required_vs_w5_speed_ratio"] = (
        x.route_proxy_required_speed_kmh / x.w5m_speed_mean.replace(0, np.nan))
    x["cur_dev_x_w5_speed"] = x.cur_dev_s * x.w5m_speed_mean
    x["cur_dev_x_w15_valid_share"] = x.cur_dev_s * x.w15m_valid_share
    x["cur_dev_x_route_speed_margin"] = x.cur_dev_s * x.route_speed_margin_kmh
    x["cur_dev_x_hour_sin"] = x.cur_dev_s * x.target_time_sin
    x["cur_dev_x_hour_cos"] = x.cur_dev_s * x.target_time_cos
    x["distance_x_valid_age"] = x.distance_to_target_m * x.valid_age_s

    for m in WINDOWS:
        x[f"w{m}m_progress_fraction"] = (
            x[f"w{m}m_target_progress_m"] / x[f"w{m}m_target_distance_first_m"].replace(0, np.nan))
        x[f"w{m}m_path_to_route_ratio"] = x[f"w{m}m_path_m"] / x.route_proxy_m.replace(0, np.nan)

    for dec in (2, 3, 4):
        x[f"same_grid_{dec}"] = (
            x[f"current_grid_{dec}"] == x[f"target_grid_{dec}"]).fillna(False).astype("int8")
    return x


def add_selected_powers(features):
    x = features.copy()
    for col in ["distance_to_target_m", "required_speed_kmh", "valid_age_s",
                "packet_age_s", "route_proxy_m"]:
        x[f"{col}_sqrt"] = np.sqrt(x[col].clip(lower=0))
    for col in ["last_valid_speed_kmh", "required_speed_kmh", "w5m_speed_mean", "w15m_speed_mean"]:
        x[f"{col}_sq"] = x[col] ** 2
    return x


TRANSFORM_PATTERNS = (
    "age_s", "distance", "path", "gap", "lag", "segment", "alt", "speed", "step",
    "eta_", "route_proxy", "cur_dev_abs", "stops_to_target", "vehicles_", "packets_per_min",
)


def select_transform_specs(train: pd.DataFrame) -> pd.Series:
    specs = {}
    for col in train.select_dtypes(include="number").columns:
        if col.endswith(("_sin", "_cos", "_sq", "_sqrt")):
            continue
        if any(tok in col for tok in ("share", "ratio", "alignment", "slope", "delta", "margin")):
            continue
        if not any(pat in col for pat in TRANSFORM_PATTERNS):
            continue
        values = train[col].dropna()
        if len(values) < 20 or values.nunique() <= 10:
            continue
        if abs(values.skew()) <= 1.5:
            continue
        specs[col] = "log" if values.min() >= 0 else "signed_log"
    return pd.Series(specs, name="transform")


def apply_distribution_transforms(df, specs):
    transformed = {}
    for col, kind in specs.items():
        if col not in df.columns:          # column may be absent in a split
            df[col] = np.nan
        if kind == "log":
            transformed[f"{col}_log1p"] = np.log1p(df[col].clip(lower=0))
        else:
            transformed[f"{col}_signed_log1p"] = signed_log1p(df[col])
    return pd.concat([df, pd.DataFrame(transformed, index=df.index)], axis=1)


PCA_CORE_COLS = [
    "cur_dev_s", "horizon_min", "last_valid_speed_kmh", "valid_age_s",
    "last_receive_lag_s", "distance_to_target_m", "required_speed_kmh",
    "route_proxy_m", "route_proxy_required_speed_kmh", "speed_margin_kmh",
    "route_speed_margin_kmh", "eta_margin_last_speed_s", "w5m_speed_mean",
    "w5m_speed_std", "w5m_target_progress_kmh", "w15m_speed_mean",
    "w15m_valid_share", "w15m_target_progress_kmh", "stop_lon", "stop_lat",
    "target_time_sin", "target_time_cos",
]


def add_pca(train_df, other_dfs, imputer=None, scaler=None, pca=None):
    """Fit PCA on train only; transform the rest."""
    from sklearn.decomposition import PCA
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    if imputer is None:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        pca = PCA(n_components=.95, random_state=SEED)
        base = imputer.fit_transform(train_df[PCA_CORE_COLS])
        train_vals = pca.fit_transform(scaler.fit_transform(base))
    else:
        train_vals = pca.transform(scaler.transform(imputer.transform(train_df[PCA_CORE_COLS])))
    cols = [f"pca_core_{i:02d}" for i in range(pca.n_components_)]
    train_df = pd.concat([train_df.reset_index(drop=True),
                          pd.DataFrame(train_vals, columns=cols)], axis=1)
    outs = []
    for df in other_dfs:
        vals = pca.transform(scaler.transform(imputer.transform(df[PCA_CORE_COLS])))
        outs.append(pd.concat([df.reset_index(drop=True), pd.DataFrame(vals, columns=cols)], axis=1))
    return train_df, outs, cols


DROP_TIME_COLS = ["last_packet_time", "last_event_time", "last_receive_time",
                  "last_gps_time", "last_valid_time", "tt_action_item_id"]


def feature_columns(df, meta):
    return [c for c in df.columns if c not in meta]


# --------------------------------------------------------------------------- #
# full pipeline
# --------------------------------------------------------------------------- #
def build_features(train_points, test_points, train_traffic, test_traffic,
                   train_schedule, test_schedule):
    """Build aligned train/test feature frames (leak-free)."""
    train_traffic = prepare_traffic(train_traffic)
    test_traffic = prepare_traffic(test_traffic)
    train_schedule = prepare_schedule(train_schedule)
    test_schedule = prepare_schedule(test_schedule)
    train_points = prepare_points(train_points)
    test_points = prepare_points(test_points)

    tr = add_base_features(train_points, train_schedule)
    te = add_base_features(test_points, test_schedule)
    tr = add_latest_state(tr, train_traffic)
    te = add_latest_state(te, test_traffic)
    tr = add_physical_features(tr)
    te = add_physical_features(te)

    tr = add_schedule_path_features(tr, make_schedule_groups(train_schedule))
    te = add_schedule_path_features(te, make_schedule_groups(test_schedule))

    tr_tg = make_traffic_groups(train_traffic)
    te_tg = make_traffic_groups(test_traffic)
    for m in WINDOWS:
        tr = tr.merge(window_features(tr, tr_tg, m), on="sample_id", how="left")
        te = te.merge(window_features(te, te_tg, m), on="sample_id", how="left")

    tr = neighbour_features(tr, make_valid_groups(train_traffic))
    te = neighbour_features(te, make_valid_groups(test_traffic))

    tr = add_cross_features(add_selected_powers(tr))
    te = add_cross_features(add_selected_powers(te))

    specs = select_transform_specs(tr)
    tr = apply_distribution_transforms(tr, specs)
    te = apply_distribution_transforms(te, specs)

    tr, (te,), _ = add_pca(tr, [te])

    tr = tr.drop(columns=[c for c in DROP_TIME_COLS if c in tr.columns])
    te = te.reindex(columns=tr.columns)
    return tr, te, list(specs.index)


In [5]:
# ---- сборка признаков (leak-free: только телеметрия с available_time <= T) ----
import time
t0 = time.time()
tr_df, te_df, transform_cols = build_features(
    train_points=train_labels, test_points=test_labels,
    train_traffic=train_traffic, test_traffic=test_traffic,
    train_schedule=train_schedule, test_schedule=test_schedule,
)
print(f'features ready in {time.time()-t0:.0f}s | train {tr_df.shape}, test {te_df.shape}')

_meta = ['sample_id', 'tr_id', 'T', 'target_stop_id', 'target_time_begin',
         'cur_dev_s', 'target_delay_s', 'target_class',
         'minute_of_day', 'target_minute_of_day',
         'target_address', 'stop_key',
         'current_grid_2', 'current_grid_3', 'current_grid_4',
         'target_grid_2', 'target_grid_3', 'target_grid_4']
num_feats = [c for c in te_df.columns
             if c not in _meta and pd.api.types.is_numeric_dtype(te_df[c])]
print('numeric features:', len(num_feats))

X_train = tr_df[num_feats].to_numpy(float)
y_train = tr_df.target_delay_s.to_numpy(float)
X_test  = te_df[num_feats].to_numpy(float)
y_test  = te_df.target_delay_s.to_numpy(float)
print('X_train', X_train.shape, '| y_train', y_train.shape,
      '| X_test', X_test.shape, '| y_test', y_test.shape)

# быстрый sanity-бейслайн: прогноз = cur_dev_s (официальный baseline ~0.40 score)
from sklearn.metrics import mean_absolute_error
_mz = float(np.mean(np.abs(y_test)))
_mae = mean_absolute_error(y_test, te_df.cur_dev_s.to_numpy(float))
print(f'baseline cur_dev_s: test MAE={_mae:.2f} (mae_zero={_mz:.2f})')


features ready in 85s | train (4434, 714), test (353, 714)
numeric features: 696
X_train (4434, 696) | y_train (4434,) | X_test (353, 696) | y_test (353,)
baseline cur_dev_s: test MAE=93.36 (mae_zero=103.34)


In [6]:
# =========================================================
# Быстрое обучение SVR / ARIMA-SARIMAX / KNN / MLP
# Использует только уже подготовленные tr_df, te_df, num_feats
# =========================================================

import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA as StatsmodelsARIMA


# ---------------------------------------------------------
# 0. Проверка, что нужные объекты уже есть
# ---------------------------------------------------------
for var in ("tr_df", "te_df", "num_feats"):
    if var not in globals():
        raise NameError(
            f"Не найден объект {var}. "
            "Сначала выполните ячейки подготовки признаков из ноутбука."
        )


# ---------------------------------------------------------
# 1. Настройки
# ---------------------------------------------------------
SEED = 143
np.random.seed(SEED)

# Сколько признаков отобрать для SVR/KNN/MLP.
# 80 — компромисс между скоростью и качеством.
K_FEATS = 80

# Если известен официальный MAE_TARGET из README/платформы, впишите сюда.
# Например: MAE_TARGET_OFFICIAL = 78.4
# Если None, оценим его из baseline cur_dev_s ~ score=0.40.
MAE_TARGET_OFFICIAL = None

# Baseline score из README для оценки MAE_TARGET, если он неизвестен.
BASELINE_SCORE_ASSUMED = 0.40


# ---------------------------------------------------------
# 2. Формируем матрицы признаков из уже готовых tr_df/te_df
# ---------------------------------------------------------
# num_feats уже исключает cur_dev_s как мета-колонку,
# но cur_dev_s — легальная подсказка на момент T, поэтому добавим его отдельно.
extra_cols = [
    c for c in ["cur_dev_s"]
    if c in tr_df.columns and c not in num_feats
]

feat_cols = list(num_feats) + extra_cols

X_train_raw = tr_df[feat_cols].to_numpy(dtype=float)
X_test_raw = te_df[feat_cols].to_numpy(dtype=float)

y_train = tr_df["target_delay_s"].to_numpy(dtype=float)
y_test = te_df["target_delay_s"].to_numpy(dtype=float)

X_train_raw = np.nan_to_num(X_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)
y_test = np.nan_to_num(y_test, nan=0.0, posinf=0.0, neginf=0.0)

y_train_median = float(np.median(y_train))


# ---------------------------------------------------------
# 3. Импутация, отбор признаков, масштабирование
# ---------------------------------------------------------
imputer = SimpleImputer(strategy="median").fit(X_train_raw)
X_train_imp = imputer.transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

X_train_imp = np.nan_to_num(X_train_imp, nan=0.0, posinf=0.0, neginf=0.0)
X_test_imp = np.nan_to_num(X_test_imp, nan=0.0, posinf=0.0, neginf=0.0)

# Убираем константные признаки
var_selector = VarianceThreshold(threshold=0.0).fit(X_train_imp)
X_train_var = var_selector.transform(X_train_imp)
X_test_var = var_selector.transform(X_test_imp)
var_cols = np.asarray(feat_cols, dtype=object)[var_selector.get_support()]

# Отбираем лучшие признаки по f_regression на train
if X_train_var.shape[1] == 0:
    # Страховка на случай, если все признаки константные
    X_train_sel = np.zeros((X_train_var.shape[0], 1), dtype=float)
    X_test_sel = np.zeros((X_test_var.shape[0], 1), dtype=float)
    selected_cols = np.asarray(["constant_zero"], dtype=object)
else:
    k = min(K_FEATS, X_train_var.shape[1])
    selector = SelectKBest(score_func=f_regression, k=k).fit(X_train_var, y_train)
    X_train_sel = selector.transform(X_train_var)
    X_test_sel = selector.transform(X_test_var)
    selected_cols = var_cols[selector.get_support()]

# Масштабируем для SVR/KNN/MLP
scaler = StandardScaler().fit(X_train_sel)
X_train_ml = scaler.transform(X_train_sel)
X_test_ml = scaler.transform(X_test_sel)

print(f"Признаков всего: {len(feat_cols)}")
print(f"После VarianceThreshold: {X_train_var.shape[1]}")
print(f"Отобрано для ML-моделей: {X_train_sel.shape[1]}")
print("Примеры отобранных признаков:", list(selected_cols[:20]))


# ---------------------------------------------------------
# 4. Метрики по README
# ---------------------------------------------------------
mae_zero = float(np.mean(np.abs(y_test)))

if "cur_dev_s" in te_df.columns:
    baseline_pred = te_df["cur_dev_s"].to_numpy(dtype=float)
    baseline_pred = np.nan_to_num(baseline_pred, nan=y_train_median, posinf=y_train_median, neginf=y_train_median)
else:
    baseline_pred = np.full_like(y_test, y_train_median, dtype=float)

baseline_mae = float(mean_absolute_error(y_test, baseline_pred))

# Оцениваем MAE_TARGET, если он не задан явно
if MAE_TARGET_OFFICIAL is not None:
    mae_target = float(MAE_TARGET_OFFICIAL)
else:
    if mae_zero > baseline_mae and BASELINE_SCORE_ASSUMED > 0:
        mae_target = mae_zero - (mae_zero - baseline_mae) / BASELINE_SCORE_ASSUMED
    else:
        mae_target = 0.0
    mae_target = float(max(0.0, mae_target))

print(f"mae_zero = {mae_zero:.3f}")
print(f"baseline cur_dev_s MAE = {baseline_mae:.3f}")
print(f"estimated/used MAE_TARGET = {mae_target:.3f}")


def score_from_mae(mae: float) -> float:
    denom = mae_zero - mae_target
    if denom <= 0:
        return np.nan
    return float(np.clip((mae_zero - mae) / denom, 0.0, 1.0))


results_rows = []


def add_result(
    name: str,
    pred: np.ndarray,
    fit_time_s: float = np.nan,
) -> float:
    pred = np.asarray(pred, dtype=float).ravel()

    if pred.size != y_test.size:
        raise ValueError(
            f"Для модели {name} длина предсказаний {pred.size} != длине теста {y_test.size}"
        )

    pred = pred.copy()
    bad = ~np.isfinite(pred)
    if bad.any():
        pred[bad] = y_train_median

    mae = float(mean_absolute_error(y_test, pred))

    results_rows.append({
        "model": name,
        "MAE": mae,
        "mae_zero": mae_zero,
        "score_est": score_from_mae(mae),
        "improvement_vs_zero_%": (
            100.0 * (mae_zero - mae) / mae_zero if mae_zero > 0 else np.nan
        ),
        "improvement_vs_baseline_%": (
            100.0 * (baseline_mae - mae) / baseline_mae if baseline_mae > 0 else np.nan
        ),
        "fit_predict_time_s": fit_time_s,
    })

    print(f"{name:45s} | MAE={mae:8.3f} | score_est={score_from_mae(mae):.3f} | time={fit_time_s:6.1f}s")
    return mae


# Простейшие baseline-предсказания
add_result("zero_prediction", np.zeros_like(y_test), fit_time_s=0.0)
add_result("cur_dev_s_baseline", baseline_pred, fit_time_s=0.0)


# ---------------------------------------------------------
# 5. SVR: 2 варианта
# ---------------------------------------------------------
svr_variants = [
    (
        "SVR_linear_C0.1",
        SVR(kernel="linear", C=0.1, tol=1e-3),
    ),
    (
        "SVR_rbf_C10_gamma_scale",
        SVR(kernel="rbf", C=10.0, gamma="scale", tol=1e-3, cache_size=500),
    ),
]

for name, model in svr_variants:
    t0 = time.time()
    model.fit(X_train_ml, y_train)
    pred = model.predict(X_test_ml)
    add_result(name, pred, fit_time_s=time.time() - t0)


# ---------------------------------------------------------
# 6. KNN: 2 варианта
# ---------------------------------------------------------
knn_variants = [
    (
        "KNN_k7_uniform_euclidean",
        KNeighborsRegressor(
            n_neighbors=7,
            weights="uniform",
            metric="euclidean",
            n_jobs=-1,
        ),
    ),
    (
        "KNN_k25_distance_manhattan",
        KNeighborsRegressor(
            n_neighbors=25,
            weights="distance",
            metric="manhattan",
            n_jobs=-1,
        ),
    ),
]

for name, model in knn_variants:
    t0 = time.time()
    model.fit(X_train_ml, y_train)
    pred = model.predict(X_test_ml)
    add_result(name, pred, fit_time_s=time.time() - t0)


# ---------------------------------------------------------
# 7. MLP: 2 варианта
# ---------------------------------------------------------
# Для MLP целевую переменную тоже лучше масштабировать
y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
y_train_scaled = y_scaler.transform(y_train.reshape(-1, 1)).ravel()

mlp_variants = [
    (
        "MLP_64_32_alpha1e-4",
        MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size="auto",
            learning_rate="adaptive",
            learning_rate_init=1e-3,
            max_iter=120,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=SEED,
        ),
    ),
    (
        "MLP_128_64_alpha1e-3",
        MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=1e-3,
            batch_size="auto",
            learning_rate="adaptive",
            learning_rate_init=5e-4,
            max_iter=150,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=SEED,
        ),
    ),
]

for name, model in mlp_variants:
    t0 = time.time()
    model.fit(X_train_ml, y_train_scaled)
    pred_scaled = model.predict(X_test_ml)
    pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    add_result(name, pred, fit_time_s=time.time() - t0)


# ---------------------------------------------------------
# 8. ARIMA / SARIMAX: базовая адаптация под табличные точки
# ---------------------------------------------------------
# Сортируем по времени прогноза T и tr_id, чтобы получить псевдо-временной ряд.
train_T = pd.to_datetime(tr_df["T"], errors="coerce").fillna(pd.Timestamp("1970-01-01")).to_numpy(dtype="datetime64[ns]")
test_T = pd.to_datetime(te_df["T"], errors="coerce").fillna(pd.Timestamp("1970-01-01")).to_numpy(dtype="datetime64[ns]")

train_id = pd.to_numeric(tr_df["tr_id"], errors="coerce").fillna(-1).to_numpy()
test_id = pd.to_numeric(te_df["tr_id"], errors="coerce").fillna(-1).to_numpy()

train_sort = np.lexsort((train_id, train_T))
test_sort = np.lexsort((test_id, test_T))

y_train_ts = y_train[train_sort]


# 8.1. ARIMA без экзогенных признаков
t0 = time.time()
try:
    arima_model = StatsmodelsARIMA(
        y_train_ts,
        order=(1, 1, 1),
    )
    arima_fit = arima_model.fit(disp=False)
    pred_ts = arima_fit.forecast(steps=len(y_test))

    pred_ts = np.asarray(pred_ts, dtype=float).ravel()
    if pred_ts.size != len(y_test):
        pred_ts = np.resize(pred_ts, len(y_test))

    pred = np.empty(len(y_test), dtype=float)
    pred[test_sort] = pred_ts

except Exception as e:
    print("ARIMA не сошёлся, используем median fallback:", e)
    pred = np.full(len(y_test), y_train_median, dtype=float)

add_result("ARIMA_1_1_1_sorted_by_T", pred, fit_time_s=time.time() - t0)


# 8.2. SARIMAX с несколькими сильными экзогенными признаками
def top_abs_corr_indices(X: np.ndarray, y: np.ndarray, k: int = 5) -> np.ndarray:
    """
    Возвращает индексы k признаков с наибольшей абсолютной корреляции с y.
    Используем только train.
    """
    y = np.asarray(y, dtype=float)
    y_centered = y - np.mean(y)
    y_norm = np.linalg.norm(y_centered)

    scores = np.zeros(X.shape[1], dtype=float)

    for j in range(X.shape[1]):
        x = np.asarray(X[:, j], dtype=float)
        x_centered = x - np.mean(x)
        x_norm = np.linalg.norm(x_centered)

        if x_norm > 0 and y_norm > 0:
            scores[j] = abs(np.dot(x_centered, y_centered) / (x_norm * y_norm))

    return np.argsort(scores)[::-1][:k]


sar_exog_k = 5
sar_exog_idx = top_abs_corr_indices(X_train_imp, y_train, k=sar_exog_k)
sar_exog_cols = np.asarray(feat_cols, dtype=object)[sar_exog_idx]

print("SARIMAX экзогенные признаки:", list(sar_exog_cols))

X_sar_train = X_train_imp[:, sar_exog_idx]
X_sar_test = X_test_imp[:, sar_exog_idx]

sar_scaler = StandardScaler().fit(X_sar_train)
X_sar_train = sar_scaler.transform(X_sar_train)
X_sar_test = sar_scaler.transform(X_sar_test)

X_sar_train_ts = X_sar_train[train_sort]
X_sar_test_ts = X_sar_test[test_sort]

t0 = time.time()
try:
    sarimax_model = sm.tsa.SARIMAX(
        y_train_ts,
        exog=X_sar_train_ts,
        order=(1, 0, 1),
        seasonal_order=(0, 0, 0, 0),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarimax_fit = sarimax_model.fit(disp=False, maxiter=30)

    pred_ts = sarimax_fit.predict(
        start=len(y_train_ts),
        end=len(y_train_ts) + len(y_test) - 1,
        exog=X_sar_test_ts,
        dynamic=False,
    )

    pred_ts = np.asarray(pred_ts, dtype=float).ravel()
    if pred_ts.size != len(y_test):
        pred_ts = np.resize(pred_ts, len(y_test))

    pred = np.empty(len(y_test), dtype=float)
    pred[test_sort] = pred_ts

except Exception as e:
    print("SARIMAX не сошёлся, используем median fallback:", e)
    pred = np.full(len(y_test), y_train_median, dtype=float)

add_result("SARIMAX_1_0_1_top5_exog_sorted_by_T", pred, fit_time_s=time.time() - t0)


# ---------------------------------------------------------
# 9. Итоговая таблица
# ---------------------------------------------------------
res_df = pd.DataFrame(results_rows).sort_values("MAE", ascending=True).reset_index(drop=True)

# Красивый вывод
show_df = res_df.copy()
for col in [
    "MAE",
    "mae_zero",
    "score_est",
    "improvement_vs_zero_%",
    "improvement_vs_baseline_%",
    "fit_predict_time_s",
]:
    if col in show_df.columns:
        show_df[col] = show_df[col].astype(float).round(3)

print("\n=== ИТОГОВАЯ ТАБЛИЦА ПО TEST ===")
print(show_df.to_string(index=False))

print("\nПояснение:")
print("- Главная метрика здесь — MAE: чем меньше, тем лучше.")
print("- score_est посчитан по формуле из README, но с оценочным MAE_TARGET.")
print("- Если знаете официальный MAE_TARGET, задайте MAE_TARGET_OFFICIAL в начале ячейки.")
print("- ARIMA/SARIMAX тут сделаны как быстрый псевдо-temporal baseline, потому что данные представлены набором прогнозных точек, а не единым регулярным временным рядом.")

Признаков всего: 697
После VarianceThreshold: 696
Отобрано для ML-моделей: 80
Примеры отобранных признаков: ['manual_fill', 'stop_lon', 'address_missing', 'schedule_rows', 'planned_speed_prev_kmh', 'time_cos', 'target_time_cos', 'cur_dev_abs', 'cur_dev_sign', 'cur_dev_log', 'cur_dev_sqrt', 'cur_dev_sq', 'cur_dev_per_min', 'speed_margin_kmh', 'scheduled_stops_to_target', 'first_future_stop_lon', 'first_future_stop_lat', 'scheduled_path_m', 'future_manual_fill_share', 'w1m_valid_share']
mae_zero = 103.337
baseline cur_dev_s MAE = 93.360
estimated/used MAE_TARGET = 78.394
zero_prediction                               | MAE= 103.337 | score_est=0.000 | time=   0.0s
cur_dev_s_baseline                            | MAE=  93.360 | score_est=0.400 | time=   0.0s
SVR_linear_C0.1                               | MAE=  77.733 | score_est=1.000 | time=   1.6s
SVR_rbf_C10_gamma_scale                       | MAE=  71.906 | score_est=1.000 | time=   1.8s
KNN_k7_uniform_euclidean                      | 

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


SARIMAX_1_0_1_top5_exog_sorted_by_T           | MAE=  85.188 | score_est=0.728 | time=   4.5s

=== ИТОГОВАЯ ТАБЛИЦА ПО TEST ===
                              model     MAE  mae_zero  score_est  improvement_vs_zero_%  improvement_vs_baseline_%  fit_predict_time_s
           KNN_k7_uniform_euclidean  57.525   103.337      1.000                 44.333                     38.384               0.048
         KNN_k25_distance_manhattan  59.626   103.337      1.000                 42.300                     36.134               0.154
                MLP_64_32_alpha1e-4  60.054   103.337      1.000                 41.885                     35.674               2.315
               MLP_128_64_alpha1e-3  63.731   103.337      1.000                 38.328                     31.737               3.754
            SVR_rbf_C10_gamma_scale  71.906   103.337      1.000                 30.416                     22.980               1.776
                    SVR_linear_C0.1  77.733   103.337      1.0